# Lane 4: Investment Research Agent

Owner: Ian Schmitt. Development notebook; finished cells are copied into `notebook.ipynb` section 4 at assembly. See `docs/` and `setup.md`.

In [ ]:
import copy
import json
import sys
import textwrap
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, "..")  # this notebook lives in dev/; make src/ importable

from src import llm, tools
from src.llm import chat, handoff


Plans its research, calls tools dynamically, reflects on its output, and learns across runs.

**Contract:** `ResearchAgent.plan(symbol) -> [{"tool", "args", "why"}]` (memory injected into the prompt, max 8 steps); `run(symbol) -> dict` (dispatches by name over `tools.TOOLS` plus `run_chain` and `route_and_analyze`, logs every call, writes the report through Lane 3's loop, with memory in the Writer's prompt too); `reflect(symbol, run_result) -> {"score", "note", "lesson"}` (takes `run()`'s output, the plan, tool log, report, and score history; takes the score and feedback from the writing loop's last grade of the final note and writes a symbol-specific note plus an optional general lesson, each tagged Plan: or Writing:, with a null note when the first draft drew no feedback); `remember(symbol, note, lesson=None)` (appends the note, if any, under the symbol and the lesson to `lessons` in `self.memory`, a dict that starts empty on every notebook run).

**Demonstration:** AAPL and NVDA twice each, interleaved (AAPL, NVDA, AAPL, NVDA), so each symbol's second run reads its own note and NVDA's first run reads AAPL's lessons. Every run with memory to read is paired with a twin: the same run by a new agent with empty memory, so a difference in plan, note, or score comes from memory (one pair per run, so it suggests rather than proves). Every call samples at the model card's recommended temperature, 1.0.

During development, import the other lanes' functions from their script exports, which live next to this notebook (regenerate with `uv run jupyter nbconvert --to script dev/chain.ipynb`):

```python
from chain import run_chain
from route import route_and_analyze
from evaluate import evaluator_optimizer
```

### Stand-ins (development only)

Until the lanes deliver on Oct 4, these four functions stand in for them with the same names, arguments, and return shapes, so the agent can be built and run now. `run_chain` and `route_and_analyze` make no LLM calls; `evaluate` and `evaluator_optimizer` make real ones so reflection has a real score to work with. When the exports exist, replace this cell with the three imports above. This cell is never copied into `notebook.ipynb`.


In [ ]:
# DEV ONLY. Same names, arguments, and return shapes as the lanes' functions.


# Lane 1 stand-in: the cached news, every article labeled neutral, no entities.
def run_chain(symbol):
    articles = [{**a, "sentiment": "neutral", "topic": "news", "entities": []} for a in tools.get_news(symbol)]
    return {"articles": articles, "summary": f"Stand-in summary of {len(articles)} {symbol} headlines."}


# Lane 2 stand-in: every item routed to news, with a placeholder analysis.
def route_and_analyze(symbol, items):
    return [{"item": item, "route": "news", "analysis": f"Stand-in analysis of: {item['title']}"} for item in items]


# Lane 3 stand-ins: one real Critic call, and at most one real revision.
def evaluate(text, criteria):
    system = (
        "You are the Critic. Check the report against each criterion. Reply as JSON: "
        '{"score": 1-10, "pass": true or false, "feedback": ["<criterion>: <what is missing or wrong>", ...]}. '
        "pass is true only if every criterion is met; feedback is empty when pass is true."
    )
    return chat(system, f"CRITERIA:\n{criteria}\n\nREPORT:\n{text}", json_mode=True, max_tokens=1500, agent="Critic")


def evaluator_optimizer(generate, criteria, max_rounds=2):
    text = generate()
    check = evaluate(text, criteria)
    history = [{"round": 0, **check}]
    if not check["pass"]:
        system = ("You are the Writer. Revise the report to address every note and change nothing else. "
                  "Reply with the revised report only, no commentary on what changed.")
        text = chat(system, f"NOTES:\n{json.dumps(check['feedback'])}\n\nREPORT:\n{text}", agent="Writer")
        check = evaluate(text, criteria)
        history.append({"round": 1, **check})
    return {"final": text, "history": history}


In [ ]:
# Every step the planner can choose, by name. The executor looks the name up here and calls it.
# Built once when this cell runs, so the lane functions must be defined (or imported) above it.
STEPS = {**tools.TOOLS, "run_chain": run_chain, "route_and_analyze": route_and_analyze}

# What the Coordinator reads when planning: what each step returns, what it costs, and its args.
# The args shown are the cached ones; any other value fetches live from Yahoo.
MENU = """\
get_prices  args {"period": "6mo"}  Daily closes plus last_close, change_pct, high, low, avg_volume. No LLM calls.
get_fundamentals  args {}  Name, sector, market cap and revenue (billions), P/E, revenue and earnings growth and
    profit margin (percent), 52-week range. No LLM calls.
get_news  args {"limit": 10}  Recent headlines with summaries, unlabeled. No LLM calls.
run_chain  args {}  Fetches the recent news itself, labels each article's sentiment and topic, extracts
    entities, and writes one summary. One LLM call per article.
route_and_analyze  args {}  Sends each article to an earnings, news, or market specialist for a short analysis.
    Two LLM calls per article. Uses run_chain's labeled articles if run_chain ran earlier in the plan,
    otherwise the raw news.
"""


In [ ]:
# The figures a note would cite, from whichever steps ran, as one JSON string. The Writer, the Critic's
# reference data, and the Reflector all read this view, so the numbers the Writer quotes are the numbers
# the Critic checks. The 125 daily closes are dropped and specialist analyses trimmed to keep prompts small.
def brief(results):
    view = {}
    if "get_prices" in results:
        view["prices"] = {k: v for k, v in results["get_prices"].items() if k not in ("dates", "close")}
    if "get_fundamentals" in results:
        view["fundamentals"] = results["get_fundamentals"]
    if "get_news" in results:
        view["headlines"] = [article["title"] for article in results["get_news"]]
    if "run_chain" in results:
        view["news_summary"] = results["run_chain"]["summary"]
        # All three labels, zeros included: Counter drops a label with no articles, and a Critic that
        # cannot find "0 positive" in the data fails a note for stating it.
        counts = Counter(a["sentiment"] for a in results["run_chain"]["articles"])
        view["sentiment_counts"] = {label: counts[label] for label in ("positive", "negative", "neutral")}
    if "route_and_analyze" in results:
        view["specialist_analyses"] = [
            {"route": row["route"], "title": row["item"]["title"], "analysis": row["analysis"][:300]}
            for row in results["route_and_analyze"]
        ]
    return json.dumps(view, indent=1)


WRITER = """\
You are the Writer on an investment research team. Write a research note of 150 to 250 words on the
stock below, using only the data given; if the data does not cover something, leave it out.
Write figures in plain words with their units ("revenue of 416.2 billion USD", "down 4.3 percent over
six months"), never as data field names. Quote figures exactly as given and compute no new ones.
Do not recommend buying, selling, or holding. If notes from earlier runs are given, follow the ones
about how to write the note; ignore the ones about which data to gather. Write in American English.
"""


# The Writer's draft. Passed to Lane 3's loop as a zero-argument generate, so the loop never needs
# to know where the text comes from. recall is the agent's memory as text, the same block the planner
# reads: a failure in the writing can only be fixed by telling the Writer.
def write_report(symbol, results, recall=""):
    memory = f"\n\nNotes from earlier runs:\n{recall}" if recall else ""
    return chat(WRITER, f"Stock: {symbol}\n\nData:\n{brief(results)}{memory}", agent="Writer")


# What the Critic grades every note against. Fixed, whatever the plan gathered: a plan that skipped
# a needed step produces a note that fails a criterion, and that failure is what reflection learns from.
CRITERIA = """\
1. States the share price change over the period, with its percent figure.
2. Cites at least one fundamental (revenue, market cap, P/E, growth, or margin) with its unit.
3. States the sentiment of recent news as counts of positive, negative, and neutral articles.
4. Names at least one specific risk drawn from the data.
5. Makes no recommendation to buy, sell, or hold.
6. Every figure in the note matches the reference data (rounding is fine).
"""


# The criteria plus the data the note was written from, so the Critic can check that a figure is
# correct, not only that one is present.
def report_criteria(results):
    return f"{CRITERIA}\nReference data:\n{brief(results)}"


# Run one plan step by name. route_and_analyze is the one step that needs another step's output: it
# takes run_chain's labeled articles if the chain ran, otherwise the raw news. Everything else takes
# the symbol plus the args from the plan.
def call_step(name, args, symbol, results):
    if name not in STEPS:
        raise ValueError(f"unknown step {name!r}; the menu has {', '.join(STEPS)}")
    if name == "route_and_analyze":
        items = results["run_chain"]["articles"] if "run_chain" in results else tools.get_news(symbol)
        return STEPS[name](symbol, items)
    return STEPS[name](symbol, **args)


In [ ]:
# The Coordinator's instructions. It sees the goal and the menu, never the Critic's criteria:
# what a good note needs is something the agent has to learn from feedback, run by run.
PLANNER = f"""\
You are the Coordinator of an investment research team. Plan the research for one stock symbol.
The team will write a short research note on the stock from the data your plan gathers.
The note states facts and never recommends buying, selling, or holding.

Steps you can run:
{MENU}
Rules:
- Choose only the steps the note needs. LLM calls cost time; do not run a step the note will not use.
- If you run route_and_analyze, run run_chain before it.
- At most 8 steps.
- Only the steps above exist. Notes about how the note is written are for the Writer and need no step.
- If a note or lesson from earlier runs leads you to add, drop, or change a step, that step's why says
  which note and what it changed.

Reply as JSON: {{"steps": [{{"tool": "<step name>", "args": {{...}}, "why": "<one sentence>"}}]}}
"""


# The Reflector's instructions. It reads the record of one run and writes the memory that the next
# run's planner and Writer both read: a note for this symbol, and a lesson only when one applies to any
# symbol. It sees the menu, so a note about the plan names a step that exists. In testing, notes written
# after a clean run and rules about what to avoid made later notes worse; the rules below stop both. It
# learns from the first draft: the writing loop usually rescues a weak draft, and a Reflector that saw
# only the final grade had nothing to learn while the next first draft repeated the same mistakes.
REFLECTOR = f"""\
You are the Reflector on an investment research team. You read the record of one research run and
write memory that will make the next run better. A failure has one of two causes: the plan gathered
the wrong data, or the Writer used the data badly.
- note: one or two sentences for the next run on this symbol. Start with "Plan:" if the plan was the
  cause or "Writing:" if the writing was. Name the criterion that failed and the change that fixes it:
  for the plan, a step from the menu below to add, drop, or change; for the writing, what the Writer
  must do. The writing loop may have rescued a failing first draft by revising it: learn from what
  the first draft got wrong (first_draft_feedback), so the next first draft is right without revision.
  If neither the first draft nor the final note drew feedback, the note is null: there is nothing to fix.
- lesson: one sentence, starting with "Plan:" or "Writing:", for a failure that would recur for any
  symbol, or null. Do not repeat a lesson that is already stored.
- Never ask for less than a criterion requires; for example, naming a risk takes a judgment, so never
  tell the Writer to avoid judgments. Say what to do, not what to avoid.
Base both only on the run record; do not invent failures it does not show.

Steps the planner can choose (no others exist):
{MENU}
Reply as JSON: {{"note": "..." or null, "lesson": "..." or null}}
"""


class ResearchAgent:
    # Memory lives on the agent and starts empty, so every notebook run is a cold start.
    # symbols holds notes per symbol; lessons holds notes that apply to any symbol.
    def __init__(self):
        self.memory = {"symbols": {}, "lessons": []}

    # Memory as text: this symbol's notes, then every general lesson. The planner and the Writer
    # both read this same block.
    def recall(self, symbol):
        notes = self.memory["symbols"].get(symbol, [])
        return (
            f"Notes from earlier runs on {symbol}:\n" + ("\n".join(f"- {n}" for n in notes) or "- none yet") + "\n\n"
            "Lessons for any symbol:\n" + ("\n".join(f"- {n}" for n in self.memory["lessons"]) or "- none yet")
        )

    # Ask the Coordinator for an ordered list of steps {"tool", "args", "why"}, with memory in the prompt.
    def plan(self, symbol):
        reply = chat(PLANNER, f"Symbol: {symbol}\n\n{self.recall(symbol)}", json_mode=True, agent="Coordinator")
        # JSON mode returns an object, so the list arrives wrapped in "steps". Unknown tool names are
        # kept: the executor logs and skips them, so a planning mistake stays visible.
        return [
            {"tool": step.get("tool"), "args": step.get("args") or {}, "why": step.get("why", "")}
            for step in reply.get("steps", [])[:8]
        ]

    # Plan, execute the plan step by step, then write the note through Lane 3's loop. Returns
    # everything reflect() needs. A failed step is logged and skipped, not raised: a run that loses
    # one source should still produce a note, and the failure is evidence for reflection.
    def run(self, symbol):
        plan = self.plan(symbol)
        results, log = {}, []
        for step in plan:
            handoff("Coordinator", step["tool"], step["why"])
            try:
                output = call_step(step["tool"], step["args"], symbol, results)
            except Exception as error:  # noqa: BLE001  any failure, from bad args to a timeout, is logged
                log.append({**step, "ok": False, "result": f"{type(error).__name__}: {error}"})
                print(f"    failed: {log[-1]['result']}")
                continue
            results[step["tool"]] = output
            log.append({**step, "ok": True, "result": str(output)[:80]})

        criteria = report_criteria(results)
        handoff("Coordinator", "Writer", f"{symbol}: data from {', '.join(results)}")
        loop = evaluator_optimizer(lambda: write_report(symbol, results, self.recall(symbol)), criteria)
        handoff("Critic", "Coordinator", f"final score {loop['history'][-1]['score']}/10")
        return {
            "symbol": symbol, "plan": plan, "log": log, "results": results, "criteria": criteria,
            "report": loop["final"], "history": loop["history"],
        }

    # Write {"score", "note", "lesson"} from run()'s output; the note is specific to this symbol, the
    # lesson (or None) applies to any symbol. The score and feedback are the writing loop's last grade
    # of the final note, so every score shown comes from one grading: in testing, a second Critic call
    # on the same note disagreed with the first (a pass, then a 4). The Critic grades the text; the
    # Reflector reads the whole run, because only the plan and the tool log can say why a note fell short,
    # and only the note beside the data it was written from can say whether the Writer was the cause.
    def reflect(self, symbol, run_result):
        final = run_result["history"][-1]
        record = {
            "symbol": symbol,
            "report": run_result["report"],
            "criteria_and_reference_data": run_result["criteria"],
            "plan": run_result["plan"],
            "tool_log": [{k: row[k] for k in ("tool", "ok", "result")} for row in run_result["log"]],
            "score_history": [{k: h[k] for k in ("round", "score", "pass")} for h in run_result["history"]],
            "first_draft_feedback": run_result["history"][0]["feedback"],
            "final_score": final["score"],
            "final_feedback": final["feedback"],
            "lessons_already_stored": self.memory["lessons"],
        }
        handoff("Critic", "Reflector", f"score {final['score']}/10, {len(final['feedback'])} feedback items")
        reply = chat(REFLECTOR, json.dumps(record, indent=1), json_mode=True, agent="Reflector")
        handoff("Reflector", "Coordinator", reply["note"] or "no note: the first draft was clean")
        return {"score": final["score"], "note": reply["note"], "lesson": reply.get("lesson") or None}

    # Append the note under this symbol and the lesson to the shared lessons list. A run whose first draft
    # was clean has no note. The exact-match check only stops verbatim repeats; the Reflector is told not to restate one.
    def remember(self, symbol, note, lesson=None):
        if note:
            self.memory["symbols"].setdefault(symbol, []).append(note)
        if lesson and lesson not in self.memory["lessons"]:
            self.memory["lessons"].append(lesson)


## Demonstration

Run the function on cached data and produce the table and figure described above. Keep printed output narrow enough for a PDF page.

Put demo code inside the `if __name__ == "__main__":` block below. It runs normally in this notebook, but not when another lane imports this notebook's `.py` export, so importing your functions never re-runs your LLM calls and plots.

In [ ]:
if __name__ == "__main__":
    # One agent, four runs, interleaved. Each symbol's second run reads the note from its first, and
    # NVDA's first run already reads the lessons from AAPL's. Every run with memory to read gets a twin:
    # the same run by a new agent with empty memory. The two differ only in memory, so the twin is the
    # control for the whole run: plan, data, note, and score.
    demo_start = len(llm.CALL_LOG)
    agent = ResearchAgent()
    runs = []
    for label, symbol in [("AAPL run 1", "AAPL"), ("NVDA run 1", "NVDA"), ("AAPL run 2", "AAPL"), ("NVDA run 2", "NVDA")]:
        print(f"\n=== {label}")
        planned_from = copy.deepcopy(agent.memory)
        try:
            result = agent.run(symbol)
            reflection = agent.reflect(symbol, result)
        except Exception as error:  # noqa: BLE001  one failed run must not end the demonstration
            print(f"{label} did not finish: {type(error).__name__}: {error}")
            continue
        agent.remember(symbol, reflection["note"], reflection["lesson"])
        twin = None
        if planned_from["symbols"] or planned_from["lessons"]:
            print(f"\n--- {label}, twin: a new agent with empty memory")
            try:
                twin = ResearchAgent().run(symbol)
            except Exception as error:  # noqa: BLE001  a failed twin leaves this run without a comparison
                print(f"{label} twin did not finish: {type(error).__name__}: {error}")
        runs.append({"label": label, "planned_from": planned_from, "twin": twin, **result})


In [ ]:
if __name__ == "__main__" and runs:
    # The steps in a plan, with arguments where a step has them and any step that failed when run
    # marked. str() keeps a malformed step visible instead of crashing the table.
    def steps(plan, log=()):
        failed = {row["tool"] for row in log if not row["ok"]}
        return [str(s["tool"]) + (f" {s['args']}" if s["args"] else "") + (" (failed)" if s["tool"] in failed else "")
                for s in plan]

    # Show a table left-aligned and in full. The export is HTML, where pandas' default right
    # alignment makes sentences hard to read.
    def show(table):
        display(table.style.set_properties(**{"text-align": "left"})
                .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

    # Number a table's rows from 1, the way a reader counts steps.
    def numbered(table, label="step"):
        table.index = range(1, len(table) + 1)
        return table.rename_axis(label)

    # Plans side by side, one column each; "-" where a plan has fewer steps.
    def plan_table(named):
        return numbered(pd.DataFrame({name: pd.Series(tools) for name, tools in named.items()}).fillna("-"))

    # The memory a run planned from: the notes on its symbol, then every general lesson.
    def memory_table(memory, symbol):
        rows = [{"applies to": symbol, "memory": note} for note in memory["symbols"].get(symbol, [])]
        rows += [{"applies to": "any symbol", "memory": lesson} for lesson in memory["lessons"]]
        return numbered(pd.DataFrame(rows, columns=["applies to", "memory"]), label="entry")

    # A research note, paragraph by paragraph, wrapped to the page, then the Critic's last grade of it.
    def print_note(heading, run):
        final = run["history"][-1]
        print(f"\n{heading} (Critic score {final['score']}/10):\n")
        for paragraph in run["report"].split("\n"):
            print(textwrap.fill(paragraph, 96) if paragraph.strip() else "")
        print("\nWhat the Critic said:")
        for item in final["feedback"] or ["no failed criteria"]:
            print(textwrap.fill(f"- {item}", 96, subsequent_indent="  "))

    # Every run that had a twin: the memory it planned from, its plan beside the twin's, why it chose
    # each step, and both notes with their grades. The only difference between the two is memory.
    for run in runs:
        if run["twin"] is None:
            continue
        print(f"\n=== {run['label']}: the memory it planned from")
        show(memory_table(run["planned_from"], run["symbol"]))
        show(plan_table({"with memory": steps(run["plan"], run["log"]),
                         "without memory (twin)": steps(run["twin"]["plan"], run["twin"]["log"])}))
        print("Why the agent with memory chose each step:")
        show(numbered(pd.DataFrame(run["plan"], columns=["tool", "why"])))
        print_note(f"{run['label']} with memory", run)
        print_note(f"{run['label']} without memory (twin)", run["twin"])


In [ ]:
if __name__ == "__main__" and runs:
    # Everything the agent learned across the runs, oldest first within each symbol, then the
    # general lessons.
    learned = [{"applies to": symbol, "memory": note}
               for symbol, notes in agent.memory["symbols"].items() for note in notes]
    learned += [{"applies to": "any symbol", "memory": lesson} for lesson in agent.memory["lessons"]]
    show(numbered(pd.DataFrame(learned), label="entry"))


In [ ]:
if __name__ == "__main__" and runs:
    # The Critic's grade of each final note, for the learning agent and for each run's memoryless twin.
    # The first run has no twin: its memory was empty, so it was its own baseline. Below: how much work
    # each agent did across the whole demonstration.
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot([run["label"] for run in runs], [run["history"][-1]["score"] for run in runs],
            marker="o", label="with memory")
    twins = [run for run in runs if run["twin"]]
    ax.plot([run["label"] for run in twins], [run["twin"]["history"][-1]["score"] for run in twins],
            marker="s", linestyle="--", label="without memory (twin)")
    ax.set(title="Critic score of the final note, per run", ylabel="score", ylim=(0, 10.5))
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    calls = llm.CALL_LOG[demo_start:]
    agents = sorted({c["agent"] for c in calls}, key=lambda name: -sum(c["agent"] == name for c in calls))
    count = [sum(c["agent"] == name for c in calls) for name in agents]
    tokens = [sum((c["prompt_tokens"] or 0) + (c["completion_tokens"] or 0) for c in calls if c["agent"] == name)
              for name in agents]
    fig, (left, right) = plt.subplots(1, 2, figsize=(9, 2.8), sharey=True)
    left.barh(agents, count)
    left.set(title="LLM calls per agent", xlabel="calls")
    left.invert_yaxis()
    right.barh(agents, tokens, color="tab:orange")
    right.set(title="Tokens per agent", xlabel="prompt + completion tokens")
    fig.tight_layout()
    plt.show()
    print(f"{len(calls)} calls, {sum(tokens):,} tokens, {sum(c['seconds'] for c in calls):.0f} s of model time")
